# Experiment: R1 to R5 Microstructure Research Pipeline

Objective:
- Run a notebook-first research loop for microstructure signals while Research Plane modules are still being built.
- Keep one reproducible artifact that maps directly to stages R1 to R5.

Done criteria:
- R1: Build a point-in-time panel from silver trades/books.
- R2: Compute notebook-level microstructure features.
- R3: Backtest one event-style signal with explicit costs.
- R4: Sweep parameters and rank robust regions.
- R5: Produce a keep/kill scorecard and export artifacts.


In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
import importlib.util
import json
import math
import random
import sys
from datetime import datetime, timezone
import numpy as np
import pandas as pd
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists():
            return candidate
    raise RuntimeError('Could not find repository root from notebook location')
try:
    NOTEBOOK_PATH = Path(__file__).resolve()
except NameError:
    NOTEBOOK_PATH = Path.cwd().resolve()
REPO_ROOT = find_repo_root(NOTEBOOK_PATH)
CORE_SRC = REPO_ROOT / 'libs' / 'core'
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(CORE_SRC) not in sys.path:
    sys.path.insert(0, str(CORE_SRC))
print('repo_root:', REPO_ROOT)
print('core_src:', CORE_SRC)
print('seed:', SEED)


In [ ]:
def module_exists(module_name: str) -> bool:
    try:
        return importlib.util.find_spec(module_name) is not None
    except ModuleNotFoundError:
        return False
STAGE_MODULES = {
    'R1': ['alpha_core.storage.duckdb_layer'],
    'R2': ['alpha_core.features.registry', 'alpha_core.features.pm_specific'],
    'R3': ['apps.research.backtester.engine'],
    'R4': ['apps.research.backtester.vec_engine', 'apps.research.backtester.sweep'],
    'R5': ['alpha_core.evaluation.evaluator', 'alpha_core.evaluation.pm_metrics'],
}
stage_status = {}
for stage, modules in STAGE_MODULES.items():
    missing = [name for name in modules if not module_exists(name)]
    stage_status[stage] = {'ready': len(missing) == 0, 'missing_modules': missing}
pd.DataFrame(stage_status).T


## Plan

Hypothesis family (microstructure):
- Spread compression plus positive order-flow pressure predicts short-horizon continuation.
- Large imbalance spikes are informative only when quoted spread is tight.

Parameters to sweep:
- Entry threshold on momentum/imbalance.
- Exit threshold.
- Maximum allowed spread.
- Holding horizon.

Metrics:
- Net PnL, Sharpe proxy, win rate, profit factor.
- Trade count and cost drag ratio.


In [ ]:
@dataclass
class NotebookConfig:
    storage_root: Path
    max_trade_rows: int = 120_000
    max_book_rows: int = 120_000
    join_tolerance_seconds: int = 60

CFG = NotebookConfig(storage_root=REPO_ROOT / 'data')
CFG


## R1 - Build Point-in-Time Microstructure Panel


In [ ]:
from alpha_core.storage.duckdb_layer import DuckDBAnalytics

analytics = DuckDBAnalytics(CFG.storage_root)
analytics.register_silver('trades')
analytics.register_silver('books')

trade_sql = (
    'SELECT instrument_id, venue, ts_event, '
    'CAST(price AS DOUBLE) AS trade_price, '
    'CAST(size AS DOUBLE) AS trade_size, side '
    'FROM silver_trades '
    'ORDER BY ts_event '
    f'LIMIT {CFG.max_trade_rows}'
)

book_sql = (
    'SELECT instrument_id, venue, ts_event, bids_json, asks_json '
    'FROM silver_books '
    'ORDER BY ts_event '
    f'LIMIT {CFG.max_book_rows}'
)

trades_df = analytics.query(trade_sql).to_pandas()
books_df = analytics.query(book_sql).to_pandas()

for frame in (trades_df, books_df):
    if 'ts_event' in frame.columns:
        frame['ts_event'] = pd.to_datetime(frame['ts_event'], utc=True)

print('trades rows:', len(trades_df))
print('books rows :', len(books_df))
trades_df.head(3)


In [ ]:
def _levels_from_json(raw: str) -> list[tuple[float, float]]:
    if raw is None or (isinstance(raw, float) and math.isnan(raw)):
        return []
    try:
        parsed = json.loads(raw) if isinstance(raw, str) else raw
    except (TypeError, json.JSONDecodeError):
        return []
    if isinstance(parsed, dict):
        if 'levels' in parsed and isinstance(parsed['levels'], list):
            parsed = parsed['levels']
        else:
            for key in ('bids', 'asks', 'yes', 'no'):
                if isinstance(parsed.get(key), list):
                    parsed = parsed[key]
                    break
    if not isinstance(parsed, list):
        return []
    levels: list[tuple[float, float]] = []
    for item in parsed:
        price = None
        size = None
        if isinstance(item, dict):
            price = item.get('price', item.get('p'))
            size = item.get('size', item.get('qty', item.get('q')))
        elif isinstance(item, (list, tuple)) and len(item) >= 2:
            price, size = item[0], item[1]
        try:
            levels.append((float(price), float(size)))
        except (TypeError, ValueError):
            continue
    return levels
def _best_bid(levels: list[tuple[float, float]]) -> tuple[float, float]:
    return max(levels, key=lambda x: x[0]) if levels else (np.nan, np.nan)
def _best_ask(levels: list[tuple[float, float]]) -> tuple[float, float]:
    return min(levels, key=lambda x: x[0]) if levels else (np.nan, np.nan)
def extract_book_metrics(row: pd.Series) -> pd.Series:
    bids = _levels_from_json(row.get('bids_json'))
    asks = _levels_from_json(row.get('asks_json'))
    best_bid_price, best_bid_size = _best_bid(bids)
    best_ask_price, best_ask_size = _best_ask(asks)
    spread = best_ask_price - best_bid_price
    denom = best_bid_size + best_ask_size
    imbalance = (best_bid_size - best_ask_size) / denom if denom and not np.isnan(denom) else np.nan
    return pd.Series(
        {
            'best_bid_price': best_bid_price,
            'best_bid_size': best_bid_size,
            'best_ask_price': best_ask_price,
            'best_ask_size': best_ask_size,
            'spread_abs': spread,
            'top_book_imbalance': imbalance,
        }
    )
if books_df.empty:
    books_enriched = books_df[['instrument_id', 'venue', 'ts_event']].copy()
    for col in ['best_bid_price', 'best_bid_size', 'best_ask_price', 'best_ask_size', 'spread_abs', 'top_book_imbalance']:
        books_enriched[col] = np.nan
else:
    book_metrics = books_df.apply(extract_book_metrics, axis=1)
    books_enriched = pd.concat([books_df[['instrument_id', 'venue', 'ts_event']], book_metrics], axis=1)
books_enriched.head(3)


In [ ]:
trades_sorted = trades_df.sort_values(['instrument_id', 'venue', 'ts_event']).reset_index(drop=True)
books_sorted = books_enriched.sort_values(['instrument_id', 'venue', 'ts_event']).reset_index(drop=True)

panel_df = pd.merge_asof(
    trades_sorted,
    books_sorted,
    on='ts_event',
    by=['instrument_id', 'venue'],
    direction='backward',
    tolerance=pd.Timedelta(seconds=CFG.join_tolerance_seconds),
)

panel_df['spread_bps'] = (panel_df['spread_abs'] / panel_df['trade_price']).replace([np.inf, -np.inf], np.nan) * 10_000

panel_summary = {
    'rows': int(len(panel_df)),
    'instruments': int(panel_df['instrument_id'].nunique()) if 'instrument_id' in panel_df else 0,
    'venues': int(panel_df['venue'].nunique()) if 'venue' in panel_df else 0,
    'missing_spread_pct': float(panel_df['spread_abs'].isna().mean()) if 'spread_abs' in panel_df else 1.0,
}
panel_summary


## R2 - Notebook-Level Feature Prototyping


In [ ]:
feature_df = panel_df.copy()
feature_df = feature_df.sort_values(['instrument_id', 'venue', 'ts_event']).reset_index(drop=True)

group_cols = ['instrument_id', 'venue']
g = feature_df.groupby(group_cols, group_keys=False)

feature_df['ret_1'] = g['trade_price'].pct_change()
feature_df['mom_5'] = g['trade_price'].pct_change(periods=5)
feature_df['mom_20'] = g['trade_price'].pct_change(periods=20)
feature_df['imbalance_ema_10'] = g['top_book_imbalance'].transform(lambda s: s.ewm(span=10, min_periods=3).mean())
feature_df['ofi_proxy_5'] = g['top_book_imbalance'].diff(5)
feature_df['fwd_ret_5'] = g['trade_price'].shift(-5) / feature_df['trade_price'] - 1.0

feature_df = feature_df.replace([np.inf, -np.inf], np.nan)

candidate_cols = [
    'mom_5',
    'mom_20',
    'spread_bps',
    'top_book_imbalance',
    'imbalance_ema_10',
    'ofi_proxy_5',
    'fwd_ret_5',
]

feature_df[candidate_cols].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T


In [ ]:
ic_rows = []
for col in ['mom_5', 'mom_20', 'top_book_imbalance', 'imbalance_ema_10', 'ofi_proxy_5']:
    valid = feature_df[[col, 'fwd_ret_5']].dropna()
    corr = valid[col].corr(valid['fwd_ret_5']) if len(valid) > 10 else np.nan
    ic_rows.append({'feature': col, 'pearson_ic_vs_fwd_ret_5': corr, 'n': len(valid)})

ic_table = pd.DataFrame(ic_rows).sort_values('pearson_ic_vs_fwd_ret_5', ascending=False)
ic_table


## R3 - Event-Style Backtest Prototype


In [ ]:
def run_event_style_backtest(
    df: pd.DataFrame,
    *,
    entry_mom: float,
    exit_mom: float,
    max_spread_bps: float,
    max_holding_obs: int,
    cost_bps_each_side: float,
) -> dict:
    work = df.sort_values('ts_event').copy()

    positions: dict[tuple[str, str], dict] = {}
    trades: list[dict] = []

    for row in work.itertuples(index=False):
        inst = getattr(row, 'instrument_id')
        venue = getattr(row, 'venue')
        key = (inst, venue)

        price = float(getattr(row, 'trade_price'))
        mom = getattr(row, 'mom_5')
        spread_bps = getattr(row, 'spread_bps')
        ts = getattr(row, 'ts_event')

        if np.isnan(price) or np.isnan(mom) or np.isnan(spread_bps):
            continue

        if key in positions:
            pos = positions[key]
            pos['holding_obs'] += 1
            should_exit = mom < exit_mom or pos['holding_obs'] >= max_holding_obs
            if should_exit:
                gross_exit = price
                net_exit = price * (1 - cost_bps_each_side / 10_000)
                gross_pnl = gross_exit - pos['gross_entry']
                net_pnl = net_exit - pos['net_entry']
                trades.append(
                    {
                        'instrument_id': inst,
                        'venue': venue,
                        'entry_ts': pos['entry_ts'],
                        'exit_ts': ts,
                        'holding_obs': pos['holding_obs'],
                        'gross_pnl': gross_pnl,
                        'net_pnl': net_pnl,
                        'cost_drag': gross_pnl - net_pnl,
                    }
                )
                del positions[key]

        if key not in positions and mom >= entry_mom and spread_bps <= max_spread_bps:
            positions[key] = {
                'entry_ts': ts,
                'gross_entry': price,
                'net_entry': price * (1 + cost_bps_each_side / 10_000),
                'holding_obs': 0,
            }

    trades_df = pd.DataFrame(trades)
    if trades_df.empty:
        return {
            'trade_count': 0,
            'gross_pnl': 0.0,
            'net_pnl': 0.0,
            'sharpe_proxy': np.nan,
            'win_rate': np.nan,
            'profit_factor': np.nan,
            'avg_net_edge': np.nan,
            'cost_drag_ratio': np.nan,
            'trades_df': trades_df,
        }

    gross_pnl = float(trades_df['gross_pnl'].sum())
    net_pnl = float(trades_df['net_pnl'].sum())
    wins = trades_df[trades_df['net_pnl'] > 0]
    losses = trades_df[trades_df['net_pnl'] < 0]

    std = float(trades_df['net_pnl'].std(ddof=1)) if len(trades_df) > 1 else np.nan
    if std and not np.isnan(std) and std > 0:
        sharpe_proxy = float(trades_df['net_pnl'].mean() / std * np.sqrt(len(trades_df)))
    else:
        sharpe_proxy = np.nan

    gross_profit = float(wins['net_pnl'].sum())
    gross_loss = float(abs(losses['net_pnl'].sum()))
    profit_factor = (gross_profit / gross_loss) if gross_loss > 0 else np.nan

    return {
        'trade_count': int(len(trades_df)),
        'gross_pnl': gross_pnl,
        'net_pnl': net_pnl,
        'sharpe_proxy': sharpe_proxy,
        'win_rate': float((trades_df['net_pnl'] > 0).mean()),
        'profit_factor': profit_factor,
        'avg_net_edge': float(trades_df['net_pnl'].mean()),
        'cost_drag_ratio': float((gross_pnl - net_pnl) / abs(gross_pnl)) if gross_pnl != 0 else np.nan,
        'trades_df': trades_df,
    }

baseline_bt = run_event_style_backtest(
    feature_df,
    entry_mom=0.004,
    exit_mom=-0.001,
    max_spread_bps=120.0,
    max_holding_obs=12,
    cost_bps_each_side=7.0,
)

{k: v for k, v in baseline_bt.items() if k != 'trades_df'}


## R4 - Parameter Sweep


In [ ]:
param_grid = {
    'entry_mom': [0.002, 0.004, 0.008],
    'exit_mom': [-0.003, -0.001, 0.0],
    'max_spread_bps': [80.0, 120.0, 180.0],
    'max_holding_obs': [8, 12, 20],
}

rows = []
for entry_mom in param_grid['entry_mom']:
    for exit_mom in param_grid['exit_mom']:
        for max_spread_bps in param_grid['max_spread_bps']:
            for max_holding_obs in param_grid['max_holding_obs']:
                summary = run_event_style_backtest(
                    feature_df,
                    entry_mom=entry_mom,
                    exit_mom=exit_mom,
                    max_spread_bps=max_spread_bps,
                    max_holding_obs=max_holding_obs,
                    cost_bps_each_side=7.0,
                )
                rows.append(
                    {
                        'entry_mom': entry_mom,
                        'exit_mom': exit_mom,
                        'max_spread_bps': max_spread_bps,
                        'max_holding_obs': max_holding_obs,
                        'trade_count': summary['trade_count'],
                        'net_pnl': summary['net_pnl'],
                        'sharpe_proxy': summary['sharpe_proxy'],
                        'win_rate': summary['win_rate'],
                        'profit_factor': summary['profit_factor'],
                        'cost_drag_ratio': summary['cost_drag_ratio'],
                    }
                )

sweep_df = pd.DataFrame(rows).sort_values(['sharpe_proxy', 'net_pnl'], ascending=False)
sweep_df.head(15)


## R5 - Evaluation and Keep/Kill Scorecard


In [ ]:
best_params = sweep_df.iloc[0].to_dict() if not sweep_df.empty else None

if best_params:
    best_bt = run_event_style_backtest(
        feature_df,
        entry_mom=float(best_params['entry_mom']),
        exit_mom=float(best_params['exit_mom']),
        max_spread_bps=float(best_params['max_spread_bps']),
        max_holding_obs=int(best_params['max_holding_obs']),
        cost_bps_each_side=7.0,
    )

    scorecard = {
        'trade_count_pass': best_bt['trade_count'] >= 30,
        'sharpe_pass': bool(np.isfinite(best_bt['sharpe_proxy']) and best_bt['sharpe_proxy'] > 0.5),
        'profit_factor_pass': bool(np.isfinite(best_bt['profit_factor']) and best_bt['profit_factor'] > 1.0),
        'cost_drag_pass': bool(np.isfinite(best_bt['cost_drag_ratio']) and best_bt['cost_drag_ratio'] < 0.5),
    }
    scorecard['decision'] = 'KEEP' if all(scorecard.values()) else 'KILL_OR_ITERATE'

    evaluation_summary = {
        'best_params': {
            'entry_mom': float(best_params['entry_mom']),
            'exit_mom': float(best_params['exit_mom']),
            'max_spread_bps': float(best_params['max_spread_bps']),
            'max_holding_obs': int(best_params['max_holding_obs']),
        },
        'metrics': {k: v for k, v in best_bt.items() if k != 'trades_df'},
        'scorecard': scorecard,
    }
else:
    evaluation_summary = {'error': 'Sweep produced no rows. Check upstream data availability.'}

evaluation_summary


In [ ]:
if {'pred_prob', 'resolved_outcome'}.issubset(feature_df.columns):
    brier = float(np.mean((feature_df['pred_prob'] - feature_df['resolved_outcome']) ** 2))
    print({'brier_score': brier})
else:
    print('Brier score skipped: add pred_prob and resolved_outcome columns when labels are available.')


## Persist Notebook Artifacts

Write a lightweight export so findings can be linked from `data/research_journal/signals/`.


In [ ]:
run_ts = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
export_dir = REPO_ROOT / 'data' / 'research_journal' / 'notebook_exports' / f'{run_ts}_r1_to_r5_microstructure'
export_dir.mkdir(parents=True, exist_ok=True)

sweep_out = export_dir / 'sweep_results.csv'
summary_out = export_dir / 'evaluation_summary.json'

if 'sweep_df' in globals() and not sweep_df.empty:
    sweep_df.to_csv(sweep_out, index=False)

with summary_out.open('w', encoding='utf-8') as f:
    json.dump(evaluation_summary, f, indent=2, default=str)

{'export_dir': str(export_dir), 'files': [p.name for p in export_dir.glob('*')]}


## Next Steps

- Convert winning notebook features into `alpha_core.features.pm_specific` classes once R2 modules are in place.
- Port the notebook simulator logic into event/vectorized engines as R3/R4 modules are implemented.
- Replace notebook scorecard with formal R5 evaluator outputs for promotion decisions.
